# Story Emotion Arc - Colab Demo

This notebook demonstrates the same functionality as the Hugging Face Space.
Paste a multi-paragraph story and see how emotions rise and fall across the narrative.

**Model:** `j-hartmann/emotion-english-distilroberta-base` (7 emotions)

In [ ]:
!pip install -q transformers torch matplotlib numpy

In [ ]:
from transformers import pipeline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)
print("Model loaded!")

In [ ]:
EMOTIONS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
COLORS = {
    "anger": "#e74c3c", "disgust": "#27ae60", "fear": "#8e44ad",
    "joy": "#f1c40f", "neutral": "#95a5a6", "sadness": "#3498db",
    "surprise": "#e67e22"
}

## Paste Your Story

Edit the text below. Separate paragraphs with blank lines.

In [ ]:
story = """The morning sun cast long shadows across the quiet village. Birds sang in the ancient oaks, and the air smelled of fresh bread. Everything felt peaceful.

Then the sirens started. At first just one thin wail from the harbor, then another and another, until the whole sky vibrated with warning.

Maria grabbed her daughter's hand and ran. Behind her she heard shouting, the crash of something heavy, glass breaking.

They reached the shelter just as rain began. Inside, families huddled together. A baby cried. An old man prayed quietly. Maria held her daughter close.

By evening the sirens had stopped. They emerged into a changed world - trees down, windows shattered, but buildings standing. Neighbors helped each other, passing water, sharing food.

Maria watched her daughter playing with other children in the rubble, already turning disaster into adventure. She felt something unexpected: hope."""

# Split into paragraphs
paras = [p.strip() for p in story.split("\n\n") if p.strip()]
print(f"Found {len(paras)} paragraphs")
for i, p in enumerate(paras):
    print(f"  P{i+1}: {p[:60]}...")

## Analyze Each Paragraph

In [ ]:
# Collect emotion scores per paragraph
arc_data = {e: [] for e in EMOTIONS}
para_results = []

for i, para in enumerate(paras[:15]):
    scores = classifier(para[:512])[0]
    score_map = {r["label"]: r["score"] for r in scores}
    for e in EMOTIONS:
        arc_data[e].append(score_map.get(e, 0))
    top = max(score_map, key=score_map.get)
    para_results.append({"index": i + 1, "top": top, "preview": para[:70], "scores": score_map})
    print(f"P{i+1} [{top}]: {para[:60]}...")

print("\nDone!")

## Emotion Arc Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(1, len(paras[:15]) + 1)

for e in EMOTIONS:
    ax.plot(x, arc_data[e], marker="o", label=e, color=COLORS[e], linewidth=2)

ax.set_xlabel("Paragraph")
ax.set_ylabel("Emotion Score")
ax.set_title("Story Emotion Arc")
ax.legend(loc="upper right", fontsize=8)
ax.set_xticks(x)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Paragraph Breakdown

In [ ]:
cards = ''
for p in para_results:
    color = COLORS.get(p["top"], "#999")
    cards += f'''
    <div style="display:flex;align-items:center;gap:8px;padding:6px 10px;
                border-left:3px solid {color};margin:4px 0;background:#fafafa;border-radius:0 6px 6px 0;">
        <span style="font-weight:700;color:#888;width:24px;">P{p["index"]}</span>
        <span style="background:{color};color:white;padding:1px 6px;border-radius:8px;font-size:0.72em;">{p["top"]}</span>
        <span style="font-size:0.85em;color:#555;">{p["preview"]}...</span>
    </div>'''

display(HTML(f'<div style="font-family:system-ui;max-width:700px;">{cards}</div>'))

## Try Your Own Text

Go back to the "Paste Your Story" cell, replace the text with your own multi-paragraph story, essay, or scene, and re-run all cells below it.

Ideas to try:
- A fairy tale (clear emotional arc from calm to danger to resolution)
- A news article (probably mostly neutral with spikes)
- Song lyrics split into verses
- Your own creative writing